# FaceForensics++ C23 — train, validate, test, compare (Colab GPU)

Trains several backbones on real FF++ face crops under the benchmark's own
**identity-level split**, then compares them on a held-out test partition.

**Before running:** Runtime ▸ Change runtime type ▸ **T4 GPU** (or better).

### What this fixes relative to the earlier work in this project

Everything measured so far used a cached 50-video subset, where the out-of-fold
AUC was 0.7307 and accuracy was bounded at 74 %. That bound is a property of
*that subset*, not of the method. Here the corpus is 2,000 videos / 64,000 face
crops and the split is by source identity, which is the protocol every published
95 %+ figure is measured under.

### The split, and why it is the whole ball game

An FF++ manipulated clip `Deepfakes/123_456.mp4` reuses the footage of
`original/123.mp4`. Splitting by clip — or worse, by frame — puts near-identical
footage on both sides of the boundary and the detector scores by recognising the
video, not the manipulation. On the small subset that inflation was measured at
**+27.9 points** for Xception RGB frames (57.64 % → 85.54 % balanced accuracy,
both halves the same representation) from the split alone. Identities 0–719
train, 720–859 validate, 860–999 test; a fake follows its *target* identity.

## 1. Environment

In [ ]:
import os, sys, json, time, math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn

SEED = 1234
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEV)
if DEV == 'cuda':
    print(torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU.')


In [ ]:
!pip -q install timm==1.0.9 scikit-learn --upgrade
import timm; print('timm', timm.__version__)


## 2. Data

Upload `ffpp_frames.zip` (produced locally by `Optimized/ffpp_prepare.py`) to
Google Drive, then run the cell below. It expects the archive to unpack to
`{train,val,test}/{real,fake}/*.jpg` plus `manifest.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/ffpp_frames.zip'   # <-- adjust if needed
ROOT = '/content/ffpp_frames'

if not os.path.exists(ROOT):
    assert os.path.exists(ZIP), f'not found: {ZIP}'
    t0 = time.time()
    !mkdir -p {ROOT} && unzip -q {ZIP} -d {ROOT}
    print(f'unzipped in {time.time()-t0:.0f}s')

# the zip may contain a single top-level folder; descend into it
if not os.path.exists(f'{ROOT}/manifest.csv'):
    subs = [d for d in os.listdir(ROOT) if os.path.isdir(f'{ROOT}/{d}')]
    if len(subs) == 1:
        ROOT = f'{ROOT}/{subs[0]}'
print('ROOT =', ROOT)

man = pd.read_csv(f'{ROOT}/manifest.csv')
print(man.groupby(['split','label']).size().to_string())
print('\nvideos per split:',
      man.groupby('split')['video'].nunique().to_dict())

# The split must be airtight: no identity may appear in two partitions.
ids = man.groupby('split')['identity'].apply(set)
for a in ids.index:
    for b in ids.index:
        if a < b:
            ov = ids[a] & ids[b]
            assert not ov, f'identity leak between {a} and {b}: {sorted(ov)[:5]}'
print('\nno identity appears in more than one split - split is clean')


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

# Normalisation is per-model, not global. Xception was trained with
# mean=std=0.5 (see the reference FF++ implementation by Roessler), while the
# EfficientNet/ConvNeXt/ViT checkpoints use ImageNet statistics. Hardcoding
# ImageNet stats for all four silently mis-scales Xception's input, so the
# values are read from each checkpoint's own config.
def data_cfg(model):
    import timm.data
    cfg = timm.data.resolve_model_data_config(model)
    return cfg['input_size'][-1], list(cfg['mean']), list(cfg['std'])

def transforms_for(size, mean, std, train):
    if train:
        return T.Compose([
            T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.RandomHorizontalFlip(0.5),
            T.ColorJitter(0.15, 0.15, 0.15, 0.02),
            T.ToTensor(), T.Normalize(mean, std),
            T.RandomErasing(p=0.25, scale=(0.02, 0.12)),
        ])
    return T.Compose([T.Resize((size, size)), T.ToTensor(),
                      T.Normalize(mean, std)])

class Frames(Dataset):
    def __init__(self, df, root, tf):
        self.df = df.reset_index(drop=True); self.root = root; self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(os.path.join(self.root, r['file'])).convert('RGB')
        return self.tf(img), int(r['label'] == 'fake'), r['video']

def loader(split, tf, bs, shuffle):
    d = Frames(man[man['split'] == split], ROOT, tf)
    return DataLoader(d, batch_size=bs, shuffle=shuffle, num_workers=2,
                      pin_memory=True, drop_last=shuffle)

print({s: int((man['split'] == s).sum()) for s in ('train','val','test')})


## 3. Models

Four families, all ImageNet-pretrained and **fully fine-tuned** — not frozen.
The frozen-feature protocol used earlier in this project was the right choice
for 40 training samples and is the wrong one for 45,000.

In [ ]:
MODELS = {
    'xception':        'legacy_xception',
    'efficientnet_b4': 'tf_efficientnet_b4.ns_jft_in1k',
    'convnext_tiny':   'convnext_tiny.fb_in22k_ft_in1k',
    'vit_small':       'vit_small_patch16_224.augreg_in21k_ft_in1k',
}

def build(name):
    m = timm.create_model(MODELS[name], pretrained=True, num_classes=2)
    return m.to(DEV)

for k, v in MODELS.items():
    print(f'{k:18s} {v}')


## 4. Train / validate / test

Two recipes per model. `reference` reproduces the published FF++ PyTorch
baseline — Adam 1e-3, StepLR(5, 0.5), no augmentation; `modern` uses AdamW 1e-4
on a OneCycle schedule with label smoothing 0.05 and augmentation. Both run the
full `EPOCHS` under AMP and keep the checkpoint with the best **validation**
video AUC — there is no early stopping, so the epoch budget is spent in full.
**The test partition is read once, after training ends.**

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             balanced_accuracy_score)

@torch.no_grad()
def infer(model, dl):
    model.eval()
    P, Y, V = [], [], []
    for x, y, v in dl:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast('cuda', enabled=(DEV=='cuda')):
            p = model(x).softmax(1)[:, 1]
        P.append(p.float().cpu().numpy()); Y.append(y.numpy()); V += list(v)
    return np.concatenate(P), np.concatenate(Y), np.array(V)

def video_level(p, y, v):
    """Aggregate frame scores to a video decision by mean log-odds, which the
    aggregation literature finds marginally better than majority vote."""
    df = pd.DataFrame({'p': np.clip(p, 1e-6, 1-1e-6), 'y': y, 'v': v})
    df['lo'] = np.log(df.p / (1 - df.p))
    g = df.groupby('v').agg(lo=('lo','mean'), y=('y','first'))
    return 1/(1+np.exp(-g.lo.values)), g.y.values

def scores(p, y, tag):
    pred = (p > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {'level': tag,
            'accuracy':  accuracy_score(y, pred)*100,
            'balanced':  balanced_accuracy_score(y, pred)*100,
            'precision': precision_score(y, pred, zero_division=0)*100,
            'recall':    recall_score(y, pred, zero_division=0)*100,
            'f1':        f1_score(y, pred, zero_division=0)*100,
            'auc':       roc_auc_score(y, p),
            'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)}

def train_one(name, epochs=20, recipe='reference', bs=32):
    """recipe='reference' reproduces the FF++ PyTorch baseline verbatim --
    Adam lr 1e-3, StepLR(step 5, gamma 0.5), batch 64, 20 epochs, no
    augmentation, mean=std=0.5 for Xception. recipe='modern' uses AdamW with
    OneCycle at 1e-4, label smoothing and the augmentation the later
    literature adds. Both are run so the comparison is against the published
    baseline rather than against my own guess at one."""
    model = build(name)
    size, mean, std = data_cfg(model)
    dl_tr = loader('train', transforms_for(size, mean, std, recipe=='modern'),
                   bs, True)
    dl_va = loader('val',   transforms_for(size, mean, std, False), bs, False)
    dl_te = loader('test',  transforms_for(size, mean, std, False), bs, False)
    print(f'  input {size}px  mean {[round(m,3) for m in mean]}  recipe {recipe}')
    if recipe == 'reference':
        opt = torch.optim.Adam(model.parameters(), lr=1e-3,
                               betas=(0.9, 0.999), eps=1e-8)
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
        per_batch_sched = False
        lossf = nn.CrossEntropyLoss()
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=1e-4, total_steps=epochs * len(dl_tr), pct_start=0.25)
        per_batch_sched = True
        lossf = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEV=='cuda'))
    best, best_state, hist = -1, None, []

    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot = 0.0; n = 0
        for x, y, _ in dl_tr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEV=='cuda')):
                loss = lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if per_batch_sched: sched.step()
            tot += loss.item()*len(y); n += len(y)
        if not per_batch_sched: sched.step()
        p, y, v = infer(model, dl_va)
        pv, yv = video_level(p, y, v)
        va = roc_auc_score(yv, pv)
        hist.append({'epoch': ep, 'train_loss': tot/n,
                     'val_frame_auc': roc_auc_score(y, p),
                     'val_video_auc': va,
                     'val_video_acc': accuracy_score(yv, (pv>0.5).astype(int))*100,
                     'sec': time.time()-t0})
        print(f"  ep{ep}/{epochs} loss {tot/n:.4f} "
              f"val-video AUC {va:.4f} acc {hist[-1]['val_video_acc']:.2f}% "
              f"[{hist[-1]['sec']:.0f}s]")
        if va > best:
            best = va
            best_state = {k: t.detach().cpu().clone()
                          for k, t in model.state_dict().items()}

    model.load_state_dict(best_state)          # best on VALIDATION only
    p, y, v = infer(model, dl_te)              # test read once, here
    pv, yv = video_level(p, y, v)
    return {'model': name, 'recipe': recipe, 'history': hist, 'best_val_video_auc': best,
            'frame': scores(p, y, 'frame'), 'video': scores(pv, yv, 'video')}


In [ ]:
EPOCHS = 20          # the reference FF++ baseline trains for 20
RECIPES = ['reference', 'modern']
# Batch is part of the recipe, not a free knob: the published FF++ baseline
# trains at 64, so 'reference' must use 64 for the word "verbatim" in
# train_one's docstring to hold. 'modern' is tuned here at 32.
BS = {'reference': 64, 'modern': 32}
results = {}
for recipe in RECIPES:
    for name in MODELS:
        key = f'{name}/{recipe}'
        print(f'\n=== {key} (batch {BS[recipe]}) ===')
        t0 = time.time()
        try:
            results[key] = train_one(name, epochs=EPOCHS, recipe=recipe,
                                     bs=BS[recipe])
            results[key]['minutes'] = (time.time()-t0)/60
            results[key]['batch_size'] = BS[recipe]
            r = results[key]['video']
            print(f"  TEST video-level: acc {r['accuracy']:.2f}%  "
                  f"AUC {r['auc']:.4f}  F1 {r['f1']:.2f}")
        except RuntimeError as e:
            print(f'  FAILED: {e}')   # usually CUDA OOM; lower bs and retry
            torch.cuda.empty_cache()

with open('/content/ffpp_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
print('\nwrote /content/ffpp_results.json')


## 5. Comparison

In [ ]:
rows = []
for k, r in results.items():
    for lvl in ('frame', 'video'):
        rows.append({'model': k, 'level': lvl,
                     **{m: r[lvl][m] for m in
                        ('accuracy','balanced','precision','recall','f1','auc')}})
cmp = pd.DataFrame(rows).sort_values(['level','accuracy'], ascending=[True,False])
print(cmp.round(3).to_string(index=False))

vid = cmp[cmp.level=='video']
if len(vid):
    b = vid.iloc[0]
    print(f"\nbest video-level: {b['model']}  "
          f"accuracy {b['accuracy']:.2f}%  AUC {b['auc']:.4f}")
    print('TARGET 95% ' + ('REACHED' if b['accuracy'] >= 95 else
                           f"NOT reached (short by {95-b['accuracy']:.2f} pts)"))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for lvl, a in zip(('frame','video'), ax):
    d = cmp[cmp.level==lvl].set_index('model')
    d[['accuracy','precision','recall','f1']].plot.bar(ax=a, rot=20,
                                                        edgecolor='k', lw=.4)
    a.axhline(95, ls=':', c='#8E24AA'); a.text(-.4, 95.6, '95% target',
                                               color='#8E24AA', fontsize=8)
    a.axhline(50, ls='--', c='#37474F', lw=.9)
    a.set_title(f'{lvl}-level, held-out test partition'); a.set_ylim(0, 105)
    a.set_ylabel('%'); a.grid(axis='y', ls=':', lw=.6)
plt.tight_layout(); plt.savefig('/content/ffpp_comparison.png', dpi=180,
                                bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for k, r in results.items():
    h = pd.DataFrame(r['history'])
    ax.plot(h.epoch, h.val_video_auc, marker='o', label=k)
ax.set_xlabel('epoch'); ax.set_ylabel('validation video AUC')
ax.set_title('Validation curve (test never touched)')
ax.legend(); ax.grid(ls=':', lw=.6)
plt.tight_layout(); plt.savefig('/content/ffpp_val_curves.png', dpi=180,
                                bbox_inches='tight'); plt.show()


In [ ]:
from google.colab import files
files.download('/content/ffpp_results.json')
files.download('/content/ffpp_comparison.png')
files.download('/content/ffpp_val_curves.png')


---
### If the target is missed

In order of expected return, and all honest:

1. **More frames per video.** 32 is conservative; the literature uses 30–300.
2. **More epochs.** `EPOCHS = 20` matches the reference baseline; the kept
   checkpoint is the best on validation, so a longer run costs mainly time.
3. **More manipulation methods.** Re-run `ffpp_prepare.py` with
   `--methods Deepfakes,Face2Face,FaceSwap,NeuralTextures` for 5,000 videos.
4. **Ensemble** the per-model video log-odds — usually worth 1–2 points.
5. **Larger backbone** (`tf_efficientnet_b7`, `convnext_base`) if VRAM allows.

What must **not** be done, because it produces a number that means nothing:
switch to a frame-level or clip-level split. On the small subset that was worth
+27 points on its own. The identity check in section 2 is there to make such a
change fail loudly rather than silently.